# 03 - Feature Engineering

## Import

In [1]:
import sys
import os

root_path = os.path.abspath(os.path.join('..'))
if root_path not in sys.path:
    sys.path.append(root_path)

import re
import polars as pl
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

import warnings
warnings.filterwarnings('ignore')

## Load dataset

In [2]:
SAVE_PATH = "../data/processed"

df_category = pl.read_parquet(f"{SAVE_PATH}/category.parquet")
df_seller = pl.read_parquet(f"{SAVE_PATH}/seller.parquet")
df_product = pl.read_parquet(f"{SAVE_PATH}/product.parquet")
df_price = pl.read_parquet(f"{SAVE_PATH}/price_offer.parquet")
df_review = pl.read_parquet(f"{SAVE_PATH}/review.parquet")
df_reviewer = pl.read_parquet(f"{SAVE_PATH}/reviewer.parquet")
df_service = pl.read_parquet(f"{SAVE_PATH}/service.parquet")
df_coupon = pl.read_parquet(f"{SAVE_PATH}/coupon.parquet")
df_offer_coupon = pl.read_parquet(f"{SAVE_PATH}/offer_coupon.parquet")
df_offer_service = pl.read_parquet(f"{SAVE_PATH}/offer_service.parquet")

In [3]:
PB_OUT = os.path.join(SAVE_PATH, "powerbi")
os.makedirs(PB_OUT, exist_ok=True)

## Mục tiêu phân tích

**Mục tiêu:** Bóc tách cấu trúc doanh thu dự phóng (Dựa trên current_price $\times$ sold_quantity) qua 3 cấp độ danh mục để tìm ra 3 "Ngách thị trường ẩn" (Niche markets - Level 3) có đóng góp doanh số vượt trội hơn 50% so với trung bình Level 1 của chúng, phục vụ báo cáo mở rộng nguồn cung trước 10/04/2026

**Tiền xử lý giá và Phân cấp danh mục**
*   **Lấy giá mới nhất:** Lọc bản ghi có `crawl_time` sau cùng cho từng `product_id`.
*   **Bóc tách Hierarchy:** Chuyển đổi chuỗi `category_path` thành các cấp bậc L1, L2, L3.

**Tính toán Doanh thu dự phóng ($P_{rev}$)**
Sử dụng công thức định lượng quy mô dòng tiền:
$$P_{rev} = \text{current\_price} \times \text{sold\_quantity}$$

**Tính toán Chỉ số hiệu suất vượt trội ($R_{out}$)**
1.  **Tổng hợp:** Tính tổng doanh thu theo ngách L3 ($\sum P_{rev, L3}$).
2.  **Tính trung bình ngành:** Tính doanh thu trung bình của các ngách L3 trong cùng một L1 ($\mu_{L1}$).
3.  **Công thức chỉ số:**
$$R_{out} = \frac{\sum P_{rev, L3}}{\mu_{L1}}$$


**Kết quả:** Xuất file `.parquet` để trực quan hóa trên Power BI.

In [4]:
latest_price = df_price.sort("crawl_time").group_by("product_id").agg(
    pl.col("current_price").last(),
    pl.col("original_price").last(),
    pl.col("discount_percent").last(),
    pl.col("offer_id").last(),
    pl.col("crawl_time").last(),
)

df_cat_levels = df_category.with_columns(
    pl.col("category_path").str.split(">").alias("path_list")
).with_columns(
    pl.col("path_list").list.get(0, null_on_oob=True).str.strip_chars().alias("L1_Category"),
    pl.col("path_list").list.get(1, null_on_oob=True).str.strip_chars().alias("L2_Category"),
    pl.col("path_list").list.get(2, null_on_oob=True).str.strip_chars().alias("L3_Category")
).drop("path_list")

df_projected_rev = (
    latest_price.join(
        df_product.select(["product_id", "category_id", "sold_quantity"]), 
        on="product_id", how="inner"
    )
    .filter(pl.col("sold_quantity").is_not_null() & (pl.col("sold_quantity") > 0))
    .with_columns(
        (pl.col("current_price") * pl.col("sold_quantity")).alias("projected_revenue")
    )
)

pbi_niche_market = (
    df_projected_rev.join(df_cat_levels, on="category_id", how="left")
    .filter(pl.col("L3_Category").is_not_null()) 
    .group_by(["L1_Category", "L2_Category", "L3_Category"])
    .agg(
        pl.col("projected_revenue").sum().alias("l3_total_revenue"),
        pl.len().alias("sku_count")
    )
    .with_columns(
        pl.col("l3_total_revenue").mean().over("L1_Category").alias("l1_avg_l3_revenue")
    )
    .with_columns(
        (pl.col("l3_total_revenue") / pl.col("l1_avg_l3_revenue")).alias("outperform_ratio")
    )
)

pbi_niche_market.write_parquet(os.path.join(PB_OUT, "pbi_category_revenue_tree.parquet"))

---

**Mục tiêu:** Theo dõi sự biến động giá (current_price / original_price) theo chuỗi thời gian thực của 5 ngành hàng lớn nhất để xác định "Chu kỳ Sale" đặc thù của từng ngành, từ đó tư vấn thời điểm tung chiến dịch Flash Sale hiệu quả nhất trước ngày 30/04/2026.

**Định danh ngành hàng và Xác định Top 5 quy mô**
*   **Mapping danh mục:** Trích xuất tên ngành hàng cấp 1 (**L1**) từ đường dẫn danh mục để đồng nhất dữ liệu phân tích.
*   **Xác định Top 5:** Thực hiện nhóm và đếm số lượng SKU trên từng ngành hàng, lọc ra 5 nhóm có quy mô hàng hóa lớn nhất sàn để đảm bảo tính đại diện thống kê.

**Tính toán Chỉ số biến động giá ($P_{ratio}$)**
Sử dụng công thức đo lường mức độ thực giảm so với giá niêm yết theo thời gian thực:
$$P_{ratio} = \frac{\text{current\_price}}{\text{original\_price}}$$

**Xây dựng Chuỗi thời gian và Tổng hợp chỉ số (Aggregation)**
1.  **Làm mịn dữ liệu:** Quy đổi thời gian thu thập về đơn vị Ngày (`record_date`) để theo dõi biến động theo chu kỳ tháng.
2.  **Tính trung vị ngành:** Sử dụng giá trị Trung vị (`median`) cho $P_{ratio}$ và `% Giảm giá` thay vì trung bình cộng để loại bỏ các nhiễu từ các sản phẩm phá giá cực đoan.
3.  **Công thức tổng hợp:**
$$P_{median} = \text{median}(P_{ratio}) \quad \text{theo từng (Ngành L1, Ngày)}$$

**Kết quả:**
*   **Xác định Chu kỳ:** Tìm ra các "vùng trũng về giá" (điểm đáy của $P_{median}$) để tư vấn thời điểm Flash Sale tối ưu.
*   **Lưu trữ:** Xuất file `pbi_price_timeseries.parquet` để phục vụ biểu đồ Small Multiples Line Chart trên Power BI.

In [5]:
cat_l1_mapping = df_category.with_columns(
    pl.col("category_path")
    .str.split(">")
    .list.get(0)
    .str.strip_chars()
    .alias("l1_category_name")
).select(["category_id", "l1_category_name"])

top5_l1_names = (
    df_product.join(cat_l1_mapping, on="category_id", how="left")
    .group_by("l1_category_name")
    .len()
    .sort("len", descending=True)
    .head(5)["l1_category_name"].to_list()
)

pbi_price_timeseries = (
    df_price
    .join(df_product.select(["product_id", "category_id"]), on="product_id", how="inner")
    .join(cat_l1_mapping, on="category_id", how="inner")
    .filter(pl.col("l1_category_name").is_in(top5_l1_names))
    
    .with_columns(
        pl.col("crawl_time").dt.date().alias("record_date"),
        (pl.col("current_price") / pl.col("original_price")).alias("price_ratio")
    )
    
    .group_by(["l1_category_name", "record_date"])
    .agg(
        pl.col("price_ratio").median().alias("median_price_ratio"),
        pl.col("discount_percent").median().alias("median_discount_pct"),
        pl.len().alias("active_listings") 
    )
    .sort(["l1_category_name", "record_date"])
)

pbi_price_timeseries.write_parquet(os.path.join(PB_OUT, "pbi_price_timeseries.parquet"))

---

**Mục tiêu:** Ứng dụng thuật toán Random Forest Classifier để phân loại tập sản phẩm thành 2 nhóm (Top 10% bán chạy và Phần còn lại). Từ đó trích xuất 3 biến số (giá, điểm đánh giá, số dịch vụ đính kèm...) có trọng số quyết định cao nhất đến doanh thu, nhằm tư vấn cho seller cách phân bổ nguồn lực vận hành trước ngày 15/04/2026.

**Tiền xử lý và Tổng hợp Đặc trưng (Feature Engineering)**

*   **Gom nhóm tiện ích:** Đếm số lượng dịch vụ (`service_count`) và mã giảm giá (`coupon_count`) đính kèm cho từng phiên bản ưu đãi.

*   **Xây dựng Master Data:** Kết nối bảng sản phẩm, giá và người bán. Tính toán biến mục tiêu **is_top_10** dựa trên ngưỡng doanh thu phân vị 90 ($Quantile_{0.9}$).

*   **Làm sạch dữ liệu:** Xử lý giá trị trống (Fill Null) cho các biến số học và loại bỏ các bản ghi không đầy đủ để đảm bảo chất lượng đầu vào cho mô hình.

In [6]:
df_services = (
    df_offer_service
    .group_by("offer_id")
    .agg(pl.len().alias("service_count"))
)

df_coupons = (
    df_offer_coupon
    .group_by("offer_id")
    .agg(pl.len().alias("coupon_count"))
)

df_latest_price = (
    df_price
    .join(df_services, on="offer_id", how="left")
    .join(df_coupons, on="offer_id", how="left")
    .with_columns([
        pl.col("service_count").fill_null(0),
        pl.col("coupon_count").fill_null(0) 
    ])
    .sort(["product_id", "crawl_time"], descending=True)
    .unique(subset=["product_id"], keep="first")
)

df_master = (
    df_product
    .join(df_latest_price, on="product_id", how="inner")
    .join(df_seller, on="seller_id", how="left")
)

df_master = df_master.with_columns(
    (pl.col("current_price") * pl.col("sold_quantity")).alias("revenue")
)

threshold_90 = df_master.select(pl.col("revenue").quantile(0.9)).item()

df_master = df_master.with_columns(
    pl.when(pl.col("revenue") >= threshold_90).then(1).otherwise(0).alias("is_top_10")
)

features = [
    "current_price", 
    "discount_percent", 
    "review_score", 
    "review_count", 
    "seller_rating", 
    "service_count",
    "coupon_count"
]

df_ml = df_master.with_columns([
    pl.col("review_score").fill_null(0),
    pl.col("review_count").fill_null(0),
    pl.col("seller_rating").fill_null(0),
    pl.col("discount_percent").fill_null(0)
]).drop_nulls(subset=features + ["is_top_10"])


X = df_ml.select(features).to_pandas()
y = df_ml.select("is_top_10").to_pandas()["is_top_10"]

**Thiết lập Mô hình và Huấn luyện (Modeling)**

1.  **Phân tách tập dữ liệu:** Chia dữ liệu theo tỷ lệ **70% Train - 15% Val - 15% Test** với phương pháp `stratify` để giữ nguyên tỷ lệ cân bằng giữa các nhóm sản phẩm.

2.  **Chuẩn hóa dữ liệu:** Sử dụng `StandardScaler` để đưa các biến có thang đo khác nhau (như giá và điểm đánh giá) về cùng một phân phối chuẩn.

3.  **Cấu hình Random Forest:** Thiết lập `class_weight='balanced'` để xử lý vấn đề lệch dữ liệu (nhóm Top 10% ít hơn nhóm còn lại) và giới hạn `max_depth=10` để tránh hiện tượng quá khớp (Overfitting).

In [7]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print(f"Kích thước tập Train (70%): {X_train.shape[0]} dòng")
print(f"Kích thước tập Val (15%): {X_val.shape[0]} dòng")
print(f"Kích thước tập Test (15%): {X_test.shape[0]} dòng")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("\nĐang huấn luyện mô hình Random Forest trên tập TRAIN...")

rf_model = RandomForestClassifier(
    n_estimators=100, 
    class_weight='balanced', 
    random_state=42, 
    max_depth=10
)

rf_model.fit(X_train_scaled, y_train)

Kích thước tập Train (70%): 212267 dòng
Kích thước tập Val (15%): 45486 dòng
Kích thước tập Test (15%): 45486 dòng

Đang huấn luyện mô hình Random Forest trên tập TRAIN...


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

**Đánh giá và Trích xuất Trọng số ($Importance\_Score$)**

*   **Kiểm chứng:** Đánh giá hiệu suất trên tập **Val** và **Test** qua các chỉ số *Precision, Recall, F1-score*. Mô hình đạt độ chính xác thực tế (Accuracy) ~93%.

*   **Trích xuất Insights:** Tính toán độ quan trọng của các đặc trưng để xác định 3 biến số có ảnh hưởng lớn nhất đến sự thành công của sản phẩm.

In [8]:
print("\n--- BÁO CÁO ĐỘ CHÍNH XÁC TRÊN TẬP VAL ---")
y_val_pred = rf_model.predict(X_val_scaled)
print(classification_report(y_val, y_val_pred))

y_test_pred = rf_model.predict(X_test_scaled)
print(classification_report(y_test, y_test_pred))

importances = rf_model.feature_importances_
df_importance = pl.DataFrame({
    "Feature_Name": features,
    "Importance_Score": importances
}).sort("Importance_Score", descending=True)


--- BÁO CÁO ĐỘ CHÍNH XÁC TRÊN TẬP VAL ---
              precision    recall  f1-score   support

           0       0.99      0.94      0.96     40937
           1       0.62      0.90      0.74      4549

    accuracy                           0.94     45486
   macro avg       0.80      0.92      0.85     45486
weighted avg       0.95      0.94      0.94     45486

              precision    recall  f1-score   support

           0       0.99      0.94      0.96     40937
           1       0.63      0.90      0.74      4549

    accuracy                           0.94     45486
   macro avg       0.81      0.92      0.85     45486
weighted avg       0.95      0.94      0.94     45486



**Kết quả:**

*   **Sản phẩm:** Xuất file `rf_feature_importance.parquet` (xếp hạng các yếu tố) và `rf_product_analysis.parquet` (kết quả dự đoán thực tế).

*   **Tư vấn:** Seller tập trung nguồn lực vào các biến có $Importance\_Score$ cao nhất (thường là lượt đánh giá và uy tín người bán) để tối ưu hóa doanh thu trước ngày 15/04/2026.

In [9]:
df_importance.write_parquet(os.path.join(PB_OUT, "rf_feature_importance.parquet"))


df_results = df_ml.with_columns(
    pl.Series("rf_prediction", rf_model.predict(X))
).select(
    ["product_id", "product_name", "revenue", "is_top_10"] + features
)

df_results.write_parquet(os.path.join(PB_OUT, "rf_product_analysis.parquet"))

---

**Mục tiêu:** Triển khai thuật toán NLP (Latent Dirichlet Allocation - LDA) trên tập dữ liệu tối thiểu 10.000 bình luận 1-2 sao để tự động gom cụm và định danh 3 nhóm nguyên nhân cốt lõi gây thất vọng (VD: Lỗi đóng gói, Giao hàng chậm), chiếm > 60% tổng phàn nàn, báo cáo trước 18/04/2026.

**Tiền xử lý và Làm sạch dữ liệu văn bản (Text Cleaning)**

*   **Lọc dữ liệu:** Trích xuất tập dữ liệu gồm hơn 18.000 bình luận tiêu cực (1-2 sao) để đảm bảo tính tập trung vào các nhóm nguyên nhân gây thất vọng.

*   **Chuẩn hóa:** Chuyển đổi về chữ thường, loại bỏ ký tự đặc biệt, chữ số và khoảng trắng thừa. 

*   **Stopwords:** Sử dụng bộ từ điển `vietnamese-stopwords.txt` để loại bỏ các từ vô nghĩa (VD: "và", "của", "là"), giúp mô hình tập trung vào các từ mang hàm lượng thông tin cao.

In [10]:
df_bad_reviews = df_review.filter(
    (pl.col("rating_score") <= 2) & 
    (pl.col("review_content").is_not_null()) & 
    (pl.col("review_content").str.len_chars() > 1)
)

print(f"Số lượng bình luận 1-2 sao cần phân tích: {df_bad_reviews.height}")

with open("../data/vietnamese-stopwords.txt", "r", encoding="utf-8") as f:
    vietnamese_stopwords = f.read().splitlines()

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_cleaned = df_bad_reviews.with_columns(
    pl.col("review_content").map_elements(clean_text, return_dtype=pl.Utf8).alias("cleaned_text")
)

df_cleaned = df_cleaned.with_columns(
    pl.col("product_id").cast(pl.Int64)
)

# Join product -> lấy product_name + seller_id
df_with_product = df_cleaned.join(
    df_product.select(["product_id", "product_name", "seller_id"]),
    on="product_id",
    how="left"
)

# Join seller -> lấy seller_name
df_with_product = df_with_product.join(
    df_seller.select(["seller_id", "seller_name"]),
    on="seller_id",
    how="left"
)

docs = df_with_product.get_column("cleaned_text").to_list()

Số lượng bình luận 1-2 sao cần phân tích: 18681


**Xây dựng Ma trận tần suất và Mô hình hóa Chủ đề (Topic Modeling)**

1.  **Vector hóa:** Sử dụng `CountVectorizer` để chuyển đổi văn bản sang ma trận tần suất từ (Document-Term Matrix), loại bỏ các từ xuất hiện quá dày đặc (>90%) hoặc quá thưa thớt (<5 bản ghi).

2.  **Thuật toán LDA:** Áp dụng mô hình LDA để tự động phân tách dữ liệu thành 4 chủ đề ẩn ($n\_components=4$) dựa trên phân phối xác suất của các từ khóa.

3.  **Công thức mô hình:**
$$P(topic|document) \times P(word|topic)$$

In [11]:
vectorizer = CountVectorizer(max_df=0.90, min_df=5, stop_words=vietnamese_stopwords)
tf_matrix = vectorizer.fit_transform(docs)

lda_model = LatentDirichletAllocation(n_components=4, random_state=42, max_iter=10)
lda_model.fit(tf_matrix)

,"n_components n_components: int, default=10Number of topics... versionchanged:: 0.19 ``n_topics`` was renamed to ``n_components``",4
,"doc_topic_prior doc_topic_prior: float, default=NonePrior of document topic distribution `theta`. If the value is None,defaults to `1 / n_components`.In [1]_, this is called `alpha`.",None
,"topic_word_prior topic_word_prior: float, default=NonePrior of topic word distribution `beta`. If the value is None, defaultsto `1 / n_components`.In [1]_, this is called `eta`.",None
,"learning_method learning_method: {'batch', 'online'}, default='batch'Method used to update `_component`. Only used in :meth:`fit` method.In general, if the data size is large, the online update will be muchfaster than the batch update.Valid options:- 'batch': Batch variational Bayes method. Use all training data in each EM update. Old `components_` will be overwritten in each iteration.- 'online': Online variational Bayes method. In each EM update, use mini-batch of training data to update the ``components_`` variable incrementally. The learning rate is controlled by the ``learning_decay`` and the ``learning_offset`` parameters... versionchanged:: 0.20 The default learning method is now ``""batch""``.",'batch'
,"learning_decay learning_decay: float, default=0.7It is a parameter that control learning rate in the online learningmethod. The value should be set between (0.5, 1.0] to guaranteeasymptotic convergence. When the value is 0.0 and batch_size is``n_samples``, the update method is same as batch learning. In theliterature, this is called kappa.",0.7
,"learning_offset learning_offset: float, default=10.0A (positive) parameter that downweights early iterations in onlinelearning. It should be greater than 1.0. In the literature, this iscalled tau_0.",10.0
,"max_iter max_iter: int, default=10The maximum number of passes over the training data (aka epochs).It only impacts the behavior in the :meth:`fit` method, and not the:meth:`partial_fit` method.",10
,"batch_size batch_size: int, default=128Number of documents to use in each EM iteration. Only used in onlinelearning.",128
,"evaluate_every evaluate_every: int, default=-1How often to evaluate perplexity. Only used in `fit` method.set it to 0 or negative number to not evaluate perplexity intraining at all. Evaluating perplexity can help you check convergencein training process, but it will also increase total training time.Evaluating perplexity in every iteration might increase training timeup to two-fold.",-1
,"total_samples total_samples: int, default=1e6Total number of documents. Only used in the :meth:`partial_fit` method.",1000000.0
,"perp_tol perp_tol: float, default=1e-1Perplexity tolerance. Only used when ``evaluate_every`` is greater than 0.",0.1


**Định danh và Phân loại Khiếu nại**

*   **Labeling:** Định danh thủ công các chủ đề dựa trên nhóm từ khóa (VD: Topic 2 chứa "đổi", "trả", "gọi", "liên" $\rightarrow$ **Khiếu nại Dịch vụ & Đổi trả**).

*   **Phân loại:** Sử dụng hàm `argmax` trên phân phối xác suất để gán nhãn chủ đề có trọng số cao nhất cho từng bình luận cụ thể.

In [12]:
feature_names = vectorizer.get_feature_names_out()
topic_keywords = {}

for topic_idx, topic in enumerate(lda_model.components_):
    top_words = [feature_names[i] for i in topic.argsort()[:-16:-1]]
    print(f"Topic {topic_idx}: {', '.join(top_words)}")

topic_names = {
    "0": "Lỗi Vận hành & Trải nghiệm sản phẩm", 
    "1": "Lỗi Hình thức & Nội dung",
    "2": "Khiếu nại Dịch vụ & Quy trình Đổi trả",
    "3": "Sản phẩm không đúng Mô tả"
}


doc_topic_distributions = lda_model.transform(tf_matrix)
dominant_topics = doc_topic_distributions.argmax(axis=1)

Topic 0: phẩm, sản, hàng, dụng, ok, sử, máy, chất, giao, gói, mua, đóng, ko, hành, ổn
Topic 1: sách, đọc, nội, dung, bìa, gói, mua, tiki, bọc, đóng, trang, hơi, ko, vọng, cũ
Topic 2: hàng, giao, tiki, mua, ko, shop, đơn, đổi, giá, tặng, tiền, đề, gửi, gọi, liên
Topic 3: ko, màu, mua, ng, mùi, bình, hơi, quảng, cáo, dán, nhựa, da, dây, nắp, đầu


**Kết quả:**

*   **Mục tiêu:** Nhận diện 3 nhóm lỗi cốt lõi (thường chiếm >80% phàn nàn) để tư vấn cải thiện vận hành trước 18/04/2026.

*   **Lưu trữ:** Xuất file `lda_bad_reviews_topics.parquet` phục vụ trực quan hóa Word Cloud và biểu đồ tỷ trọng trên Power BI.

In [13]:
df_final = df_with_product.with_columns([
    pl.Series("topic_id", dominant_topics),
    pl.Series("topic_id_temp", dominant_topics)
      .cast(pl.String)
      .replace(topic_names)
      .alias("topic_name")
])

df_export = df_final.select([
    "review_id",
    "product_id",
    "product_name",
    "seller_name",
    "rating_score",
    "review_content",
    "cleaned_text",
    "topic_name"
])

out_path = os.path.join(PB_OUT, "lda_bad_reviews_topics.parquet")
df_export.write_parquet(out_path)

In [14]:
lda_review = pl.read_parquet(f"{PB_OUT}/lda_bad_reviews_topics.parquet")
lda_review

review_id,product_id,product_name,seller_name,rating_score,review_content,cleaned_text,topic_name
str,i64,str,str,i8,str,str,str
"""7b342300-05c""",10005245,"""Sách Đàn Ông Sao Hỏa Đàn Bà Sa…","""Tiki Trading""",1,"""Sách rất đẹp, giao hàng cực nh…","""sách rất đẹp giao hàng cực nha…","""Khiếu nại Dịch vụ & Quy trình …"
"""ea4a4c70-738""",10005245,"""Sách Đàn Ông Sao Hỏa Đàn Bà Sa…","""Tiki Trading""",1,"""Đây là lần đầu mình mua sách ở…","""đây là lần đầu mình mua sách ở…","""Lỗi Hình thức & Nội dung"""
"""03c283fd-95d""",1000534,"""Sữa Bột Anlene Hương Vanilla (…","""Tiki Trading""",1,"""Nhận gói sữa xong muốn vứt luô…","""nhận gói sữa xong muốn vứt luô…","""Khiếu nại Dịch vụ & Quy trình …"
"""2b61f6dd-2f5""",1000534,"""Sữa Bột Anlene Hương Vanilla (…","""Tiki Trading""",1,"""giao hàng không đúng như trên …","""giao hàng không đúng như trên …","""Khiếu nại Dịch vụ & Quy trình …"
"""7192b284-439""",1000534,"""Sữa Bột Anlene Hương Vanilla (…","""Tiki Trading""",1,"""Tôi đặt sản phẩm này cho người…","""tôi đặt sản phẩm này cho người…","""Lỗi Vận hành & Trải nghiệm sản…"
…,…,…,…,…,…,…,…
"""20daabf1-30b""",99862768,"""Combo 2 cuốn: Luật Tâm Thức + …","""AHABOOKS""",1,"""Tuyệt vời""","""tuyệt vời""","""Lỗi Hình thức & Nội dung"""
"""2be0ad46-962""",99862768,"""Combo 2 cuốn: Luật Tâm Thức + …","""AHABOOKS""",1,"""Vuviviviv""","""vuviviviv""","""Lỗi Vận hành & Trải nghiệm sản…"
"""9ec51249-2f0""",99862768,"""Combo 2 cuốn: Luật Tâm Thức + …","""AHABOOKS""",1,"""Tại sao lúc về quyển Luật tâm …","""tại sao lúc về quyển luật tâm …","""Lỗi Hình thức & Nội dung"""


---

**Mục tiêu:** Phân tích cơ cấu giá và mức giảm giá của các sản phẩm trên Tiki trong giai đoạn thu thập dữ liệu để xác định 5 nhóm ngành hàng có giá trung vị và tỷ lệ giảm giá cao nhất, đồng thời giải thích ít nhất 60\% sự khác biệt về mức giá giữa các nhóm sản phẩm trước ngày 30/04/2026.

**Tiền xử lý và Làm sạch dữ liệu giá**

*   **Lấy giá mới nhất:** Sử dụng `tail(1)` sau khi sắp xếp theo thời gian để trích xuất mức giá hiện hành duy nhất cho mỗi sản phẩm, loại bỏ các giá trị lỗi hoặc bằng 0.

*   **Chuẩn hóa phân phối:** Áp dụng phép biến đổi logarit `log1p()` cho cột giá (`current_price`) để giảm độ lệch (skewness) của dữ liệu, đưa về phân phối chuẩn giúp phép thử thống kê chính xác hơn.

In [15]:
df_price_latest = (
    df_price
    .filter(
        pl.col("product_id").is_not_null() &
        pl.col("current_price").is_not_null() &
        (pl.col("current_price") > 0)
    )
    .sort(["product_id", "crawl_time"])
    .group_by("product_id")
    .tail(1)
    .select(["product_id", "current_price"])
)

df = (
    df_product
    .select(["product_id", "category_id"])
    .join(
        df_category.select(["category_id", "category_name"]),
        on="category_id",
        how="left"
    )
    .join(
        df_price_latest,
        on="product_id",
        how="left"
    )
    .filter(
        pl.col("category_name").is_not_null() &
        pl.col("current_price").is_not_null() &
        (pl.col("current_price") > 0)
    )
    .with_columns(
        pl.col("current_price").log1p().alias("log_price")
    )
)

**Tính toán Phân rã phương sai (Sum of Squares)**

Sử dụng phương pháp phân tích phương sai để đo lường mức độ ảnh hưởng của yếu tố ngành hàng lên giá bán:

1.  **SST (Total Sum of Squares):** Tổng biến thiên toàn bộ của mức giá so với giá trung bình tổng thể ($Grand\ Mean$).

2.  **SSE (Sum of Squared Errors):** Tổng biến thiên còn lại bên trong nội bộ từng ngành hàng sau khi đã tính toán trung bình nhóm.

3.  **Công thức các thành phần:**

$$SST = \sum (y_i - \bar{y})^2 \quad ; \quad SSE = \sum (y_i - \bar{y}_{group})^2$$

**Xác định chỉ số giải thích ($R^2$)**

Sử dụng chỉ số R-squared để định lượng tỷ lệ sự khác biệt về giá được giải thích bởi yếu tố phân loại ngành hàng:
$$R^2 = 1 - \frac{SSE}{SST}$$

**Kết quả:**

*   **Kiểm chứng:** Xác định xem chỉ số $R^2$ có đạt ngưỡng **> 0.6** (tức ngành hàng giải thích được > 60% sự biến động giá) như mục tiêu đề ra hay không.

*   **Insights:** Nhận diện 5 nhóm ngành có giá trung vị và tỷ lệ giảm giá cao nhất để định vị chiến lược giá cho Seller trước ngày 30/04/2026.

In [16]:
grand_mean = df.select(pl.col("log_price").mean()).item()

# Tính SST = tổng biến thiên toàn bộ
sst = df.select(
    ((pl.col("log_price") - grand_mean) ** 2).sum().alias("sst")
).item()

# Tính mean theo từng ngành
group_means = (
    df.group_by("category_name")
    .agg(pl.col("log_price").mean().alias("group_mean"))
)

# Tính SSE = tổng biến thiên còn lại sau khi giải thích bởi ngành hàng
df_with_mean = df.join(group_means, on="category_name", how="left")

sse = df_with_mean.select(
    ((pl.col("log_price") - pl.col("group_mean")) ** 2).sum().alias("sse")
).item()

# R-squared
r2 = 1 - sse / sst

print(f"R-squared = {r2:.4f}")

R-squared = 0.6458
